In [1]:
%pip install --upgrade --quiet langchain langchain-neo4j langchain-openai langchain-experimental neo4j python-dotenv pyvis

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
from dotenv import load_dotenv
from langchain_core.documents import Document
from pyvis.network import Network

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")

In [3]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_openai import OpenAI

llm = OpenAI(temperature=0, model_name="gpt-4o-mini")
llm_transformer = LLMGraphTransformer(llm)

In [4]:
docs = [
    Document(page_content="Albert Einstein was a theoretical physicist born in Ulm, Germany."),
    Document(page_content="Albert Einstein developed the Theory of Relativity."),
    Document(page_content="Albert Einstein won the Nobel Prize in Physics in 1921."),
    Document(page_content="Einstein worked at the Swiss Patent Office in Bern."),

    Document(page_content="Marie Curie was a physicist and chemist born in Warsaw, Poland."),
    Document(page_content="Marie Curie discovered Radium and Polonium."),
    Document(page_content="Marie Curie worked at the University of Paris."),
    Document(page_content="Marie Curie won the Nobel Prize in Chemistry in 1911."),

    Document(page_content="Isaac Newton formulated the laws of motion and universal gravitation."),
    Document(page_content="Isaac Newton published Principia Mathematica in 1687."),
    Document(page_content="Isaac Newton served as President of the Royal Society."),

    Document(page_content="The Theory of Relativity influenced modern cosmology."),
    Document(page_content="Radioactivity led to advances in nuclear physics."),
    Document(page_content="The University of Paris is located in Paris, France."),
    Document(page_content="The Royal Society is based in London, England.")
]



In [5]:
# Extract graph documents
graph_docs = await llm_transformer.aconvert_to_graph_documents(docs)

In [6]:
print(f"Nodes: {graph_docs[0].nodes}")
print(f"Relationships: {graph_docs[0].relationships}")

Nodes: [Node(id='theory of relativity', type='Theory', properties={}), Node(id='Ulm', type='Location', properties={}), Node(id='Nobel Prize in Physics', type='Award', properties={}), Node(id='Albert Einstein', type='Person', properties={}), Node(id='Germany', type='Location', properties={}), Node(id='explanation of the photoelectric effect', type='Achievement', properties={})]
Relationships: [Relationship(source=Node(id='Albert Einstein', type='Person', properties={}), target=Node(id='Ulm', type='Location', properties={}), type='BORN_IN', properties={}), Relationship(source=Node(id='Albert Einstein', type='Person', properties={}), target=Node(id='Germany', type='Location', properties={}), type='BORN_IN', properties={}), Relationship(source=Node(id='Albert Einstein', type='Person', properties={}), target=Node(id='theory of relativity', type='Theory', properties={}), type='DEVELOPED', properties={}), Relationship(source=Node(id='Albert Einstein', type='Person', properties={}), target=Nod

In [7]:

def visualize_graph_documents(
    graph_docs,
    output_html="knowledge_graph.html",
    notebook=False
):
    """
    Visualize LangChain GraphDocuments in an interactive HTML file.

    Args:
        graph_docs (list[GraphDocument]): Output from LLMGraphTransformer
        output_html (str): Path to save the HTML file
        notebook (bool): Set True if running in Jupyter
    """

    net = Network(
        height="750px",
        width="100%",
        directed=True,
        notebook=notebook
    )

    added_nodes = set()

    for graph_doc in graph_docs:
        # Add nodes
        for node in graph_doc.nodes:
            node_id = node.id
            if node_id not in added_nodes:
                net.add_node(
                    node_id,
                    label=node_id,
                    title=f"Type: {node.type}"
                )
                added_nodes.add(node_id)

        # Add relationships (edges)
        for rel in graph_doc.relationships:
            net.add_edge(
                rel.source.id,
                rel.target.id,
                label=rel.type,
                title=rel.type
            )

    net.write_html(output_html)
    print(f"Graph visualization saved to {output_html}")


In [8]:
visualize_graph_documents(graph_docs)

Graph visualization saved to knowledge_graph.html


In [9]:
llm_transformer_filtered = LLMGraphTransformer(llm,
                                      allowed_nodes=["Person"],allowed_relationships=["awarded"])
graph_docs_filtered = await llm_transformer_filtered.aconvert_to_graph_documents(docs)
visualize_graph_documents(graph_docs_filtered, output_html="knowledge_graph_filtered.html")

Graph visualization saved to knowledge_graph_filtered.html


In [10]:
from langchain_neo4j import Neo4jGraph
graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD")
)

In [11]:
graph.add_graph_documents(graph_docs)

In [16]:
query = "MATCH (n:Institution) RETURN n LIMIT 25;"

# If your Neo4jGraph uses 'query' method
results = graph.query(query)

# Often returns a pandas DataFrame directly
print(results)


[{'n': {'id': 'University of Paris'}}]
